# CSI800 V55 Anchor Consensus Rerank & Health Monitor

## 实验目的

本实验不新增大模型，也不重新定义 label。它只验证一个问题：在强 direct anchor 已经给出全域排序以后，top30 候选池内部是否还能通过简单、可解释的因子家族共识排序改善 top10/top20，并同时输出模型健康监控指标。

## 核心假设

1. `alpha_1m` + LGB direct anchor 仍然是主线 baseline。
2. rerank 只有在 anchor top30 对真实 topK 有足够 recall gap 时才有价值。
3. 第一版 rerank 应该保持简单：不训练新模型，只在 top30 内做 factor-family vote / defensive vote。
4. 如果共识排序不能稳定优于 anchor top10/top20，它就只作为诊断，不进入回测模型。

## 输入要求

本 notebook 默认先读取固定 score panel：`csi800_v55_anchor_score_panel.csv`。如果该文件不存在，会用 `train_csi800_factor_v40_data_enhancement.csv` + `model_candidate_v46_lgb_direct_hybrid_l2_ff10_2019_2025q1_legacy_unsealed.pkl` 自动生成。

必须包含：

- `rebalance_date`
- `stock`
- `anchor_score`：例如 v46/v410 direct 模型线上同口径分数
- `alpha_1m`：下一月超额收益 label，用于离线评估

建议包含：

- `raw_return_1m`
- `benchmark_csi800_1m`
- `board` 或 `board_main/board_chinext/board_star`
- `industry` 或 `industry_bucket`
- 当前固定因子池的 JQ factor columns

## 数据来源

第一优先级：`csi800_v55_anchor_score_panel.csv`，已经带 `anchor_score`。

第二优先级：`train_csi800_factor_v40_data_enhancement.csv` + anchor pkl，在 notebook 内按 bundle 的 `base_feature_cols` 和 `base_fill_values` 直接打分生成 `anchor_score`。

如果当前目录没有 `train_csi800_factor_v40_data_enhancement.csv`，可以在 config cell 里设置 `REBUILD_DATA=True`，在聚宽研究环境直接重建该 CSV。

这个设计是为了让研究评估和线上 pkl 加载口径保持一致，而不是手工拼一个分数文件。

## 输出

输出目录：`csi800_ml_v55_anchor_consensus_rerank_monitor_outputs`

- `v55_summary.csv`
- `v55_monthly.csv`
- `v55_health_monitor.csv`
- `v55_topk_recall.csv`
- `v55_vote_attribution.csv`
- `v55_latest_targets.csv`
- `v55_config.json`

In [ ]:
import os
import json
import math
import pickle
import gc
import datetime
import numpy as np
import pandas as pd

try:
    from jqdata import *
    from jqfactor import get_factor_values
except Exception:
    # Local syntax checks do not have JoinQuant APIs. Rebuild path runs only inside JoinQuant research.
    pass

# =========================
# Fixed config
# =========================
SCORE_PANEL_PATH = "csi800_v55_anchor_score_panel.csv"
RAW_DATA_PATH = "train_csi800_factor_v40_data_enhancement.csv"
ANCHOR_MODEL_PATH = "model_candidate_v46_lgb_direct_hybrid_l2_ff10_2019_2025q1_legacy_unsealed.pkl"
AUTO_BUILD_SCORE_PANEL = True

# Raw dataset rebuild switch. Keep False normally; set True inside JoinQuant research to regenerate RAW_DATA_PATH.
REBUILD_DATA = False
USE_REBUILT_DATA_FOR_TRAINING = True
REBUILD_DATA_START = "2019-01-01"
REBUILD_DATA_END_FOR_LABEL = "2026-05-31"
REBUILD_DATA_OUTPUT_PATH = RAW_DATA_PATH
REBUILD_MANIFEST_PATH = "train_csi800_factor_v40_data_enhancement_manifest.csv"
REBUILD_FORCE_OVERWRITE = False

OUT_DIR = "csi800_ml_v55_anchor_consensus_rerank_monitor_outputs"
BENCHMARK = "000906.XSHG"

DATE_COL = "rebalance_date"
STOCK_COL = "stock"
ANCHOR_SCORE_COL = "anchor_score"
TARGET_ALPHA_COL = "alpha_1m"
RETURN_COL = "raw_return_1m"              # optional; falls back to TARGET_ALPHA_COL
BENCHMARK_COL = "benchmark_csi800_1m"     # optional
INDUSTRY_COL = "industry_bucket"          # optional
BOARD_COL = "board"                       # optional

TOP_N_CANDIDATES = 30
VOTE_TOP_N = 10
PORTFOLIO_SIZES = [10, 20]
RECENT_WINDOW_MONTHS = 6

os.makedirs(OUT_DIR, exist_ok=True)

CONFIG = {
    "score_panel_path": SCORE_PANEL_PATH,
    "raw_data_path": RAW_DATA_PATH,
    "anchor_model_path": ANCHOR_MODEL_PATH,
    "auto_build_score_panel": AUTO_BUILD_SCORE_PANEL,
    "rebuild_data": REBUILD_DATA,
    "rebuild_data_start": REBUILD_DATA_START,
    "rebuild_data_end_for_label": REBUILD_DATA_END_FOR_LABEL,
    "date_col": DATE_COL,
    "stock_col": STOCK_COL,
    "anchor_score_col": ANCHOR_SCORE_COL,
    "target_alpha_col": TARGET_ALPHA_COL,
    "return_col": RETURN_COL,
    "benchmark_col": BENCHMARK_COL,
    "benchmark": BENCHMARK,
    "top_n_candidates": TOP_N_CANDIDATES,
    "vote_top_n": VOTE_TOP_N,
    "portfolio_sizes": PORTFOLIO_SIZES,
    "recent_window_months": RECENT_WINDOW_MONTHS,
}

with open(os.path.join(OUT_DIR, "v55_config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2, sort_keys=True)

In [ ]:
# =========================
# Factor families
# =========================
FACTOR_GROUPS = {
    "value_cashflow": [
        "cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield",
        "sales_to_price_ratio", "cash_earnings_to_price_ratio", "earnings_to_price_ratio",
    ],
    "quality_profit": [
        "roe_ttm", "roa_ttm", "gross_profit_ttm", "operating_profit_to_total_profit",
        "net_operate_cash_flow_to_total_liability", "net_operating_cash_flow_coverage",
        "adjusted_profit_to_total_profit", "operating_profit_per_share",
        "net_operate_cash_flow_per_share", "total_operating_revenue_per_share",
    ],
    "growth_balance": [
        "ACCA", "growth", "net_working_capital", "super_quick_ratio", "MLEV",
        "debt_to_equity_ratio", "debt_to_tangible_equity_ratio",
    ],
    "momentum_risk": [
        "momentum", "Rank1M", "sharpe_ratio_60", "Variance20", "beta",
        "Skewness20", "Kurtosis20", "Kurtosis60",
    ],
    "technical_volume": [
        "liquidity", "MFI14", "DAVOL10", "VOL10", "VMACD", "VOSC",
    ],
}

# 1 means higher is better; -1 means lower is better.
# This is intentionally conservative and easy to edit after reviewing attribution.
FACTOR_DIRECTIONS = {
    "cash_flow_to_price_ratio": 1,
    "book_to_price_ratio": 1,
    "earnings_yield": 1,
    "sales_to_price_ratio": 1,
    "cash_earnings_to_price_ratio": 1,
    "earnings_to_price_ratio": 1,
    "roe_ttm": 1,
    "roa_ttm": 1,
    "gross_profit_ttm": 1,
    "operating_profit_to_total_profit": 1,
    "net_operate_cash_flow_to_total_liability": 1,
    "net_operating_cash_flow_coverage": 1,
    "adjusted_profit_to_total_profit": 1,
    "operating_profit_per_share": 1,
    "net_operate_cash_flow_per_share": 1,
    "total_operating_revenue_per_share": 1,
    "ACCA": 1,
    "growth": 1,
    "net_working_capital": 1,
    "super_quick_ratio": 1,
    "MLEV": -1,
    "debt_to_equity_ratio": -1,
    "debt_to_tangible_equity_ratio": -1,
    "momentum": 1,
    "Rank1M": 1,
    "sharpe_ratio_60": 1,
    "Variance20": -1,
    "beta": -1,
    "Skewness20": 1,
    "Kurtosis20": 1,
    "Kurtosis60": 1,
    "liquidity": -1,
    "MFI14": 1,
    "DAVOL10": -1,
    "VOL10": -1,
    "VMACD": 1,
    "VOSC": 1,
}

GROUP_SCORE_COLS = ["group_score_" + k for k in FACTOR_GROUPS]

In [ ]:
# =========================
# V46/V4 raw dataset feature columns
# =========================
BASE_FACTOR_COLS = [
    "cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield", "sales_to_price_ratio",
    "cash_earnings_to_price_ratio", "earnings_to_price_ratio", "roe_ttm", "roa_ttm",
    "gross_profit_ttm", "operating_profit_to_total_profit", "net_operate_cash_flow_to_total_liability",
    "net_operating_cash_flow_coverage", "adjusted_profit_to_total_profit", "ACCA", "growth",
    "net_working_capital", "operating_profit_per_share", "net_operate_cash_flow_per_share",
    "total_operating_revenue_per_share", "super_quick_ratio", "MLEV", "debt_to_equity_ratio",
    "debt_to_tangible_equity_ratio", "momentum", "Rank1M", "sharpe_ratio_60", "Variance20",
    "liquidity", "beta", "ATR6", "MFI14", "DAVOL10", "VOL10", "VMACD", "VOSC",
    "Skewness20", "Kurtosis20",
]

HYBRID_LIGHT_EXTRA_COLS = [
    "liq_money_ratio_20_60",
    "liq_paused_count_20",
    "px_close_to_ma60",
    "px_drawdown_60",
    "ts_cash_flow_to_price_ratio_rank_mean_3m",
    "ts_Rank1M_rank_chg_1m",
]

BASE_PARAMS_FF10 = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 200,
    "feature_fraction": 1.0,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 0.3,
    "verbose": -1,
}

CANDIDATE_COLS = BASE_FACTOR_COLS + HYBRID_LIGHT_EXTRA_COLS


In [ ]:
# =========================
# Optional V46/V4 raw data rebuild block
# =========================
# Default path: REBUILD_DATA=False, use the existing RAW_DATA_PATH CSV.
# Rebuild path: set REBUILD_DATA=True inside JoinQuant research; this will regenerate the monthly CSI800 V4/V46 dataset.

UNIVERSE_NAME = "CSI800"
UNIVERSE_INDEX = "000906.XSHG"
MIN_LISTING_DAYS = 180
V4_DATA_START = REBUILD_DATA_START
V4_DATA_END_FOR_LABEL = REBUILD_DATA_END_FOR_LABEL
V4_DATA_FILE = REBUILD_DATA_OUTPUT_PATH

V4_PRICE_PATH_COLS = [
    "px_ret_5", "px_ret_20", "px_ret_60", "px_ret_120",
    "px_close_to_ma20", "px_close_to_ma60", "px_ma20_to_ma60",
    "px_volatility_20", "px_volatility_60", "px_drawdown_20", "px_drawdown_60", "px_drawdown_120",
    "px_up_day_ratio_20", "px_new_high_distance_60", "px_new_low_distance_60",
    "px_skew_20", "px_kurt_20",
]

TRADE_LIQUIDITY_COLS = [
    "liq_money_mean_20", "liq_money_mean_60", "liq_money_ratio_20_60",
    "liq_volume_mean_20", "liq_volume_ratio_20_60",
    "liq_amplitude_mean_20", "liq_amplitude_mean_60",
    "liq_paused_count_20", "liq_paused_count_60",
    "liq_low_money_days_20", "liq_limit_up_count_20", "liq_limit_down_count_20", "liq_one_price_limit_count_20",
]

CONTEXT_COLS = [
    "ctx_industry_ret_20", "ctx_industry_ret_60",
    "ctx_stock_minus_industry_ret_20", "ctx_stock_minus_industry_ret_60",
    "ctx_stock_rank_industry_ret_20", "ctx_stock_rank_industry_volatility_20",
    "ctx_market_ret_20", "ctx_market_ret_60", "ctx_market_volatility_20",
]

CORE_TEMPORAL_FACTORS = [
    "book_to_price_ratio", "earnings_yield", "cash_flow_to_price_ratio",
    "Rank1M", "sharpe_ratio_60", "VOSC", "MFI14",
]

def chunks(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def require_joinquant_api():
    # Directly probe the JoinQuant API instead of inspecting notebook namespaces.
    try:
        get_trade_days(end_date="2019-01-02", count=1)
    except NameError:
        raise RuntimeError("data rebuild requires JoinQuant research runtime: get_trade_days is not available")
    except Exception:
        # API exists; date/account related errors should surface in the actual caller.
        pass


def get_period_date(period, start_date, end_date):
    require_joinquant_api()
    trade_days = pd.to_datetime(get_trade_days(start_date=start_date, end_date=end_date))
    if len(trade_days) == 0:
        return []
    if period != "M":
        raise ValueError("V4 data pipeline only supports monthly period M")

    dates = []
    last_key = None
    for d in trade_days:
        key = d.strftime("%Y-%m")
        if key != last_key:
            dates.append(d.strftime("%Y-%m-%d"))
            last_key = key
    return dates


def get_previous_trade_date(date):
    require_joinquant_api()
    trade_days = pd.to_datetime(get_trade_days(end_date=date, count=2))
    if len(trade_days) < 2:
        return None
    return trade_days[-2].strftime("%Y-%m-%d")


def delect_stop(stocks, begin_date, n=180):
    stock_list = []
    begin_dt = pd.Timestamp(begin_date).to_pydatetime()
    for stock in stocks:
        info = get_security_info(stock)
        if info is None:
            continue
        if info.start_date <= (begin_dt - datetime.timedelta(days=n)).date():
            stock_list.append(stock)
    return stock_list


def filter_paused_stock_by_date(stock_list, date):
    if len(stock_list) == 0:
        return []
    try:
        paused_df = get_price(
            stock_list,
            end_date=date,
            frequency="daily",
            fields=["paused"],
            count=1,
            skip_paused=False,
            panel=False,
            fill_paused=True,
        )
    except Exception:
        return stock_list

    if paused_df is None or paused_df.empty or "paused" not in paused_df.columns:
        return stock_list

    paused_map = paused_df.groupby("code")["paused"].last()
    return [
        stock for stock in stock_list
        if (stock not in paused_map.index) or (not bool(paused_map.loc[stock]))
    ]


def get_stock(stock_pool, feature_date):
    require_joinquant_api()
    if stock_pool == "CSI800":
        stock_list = get_index_stocks(UNIVERSE_INDEX, feature_date)
    elif stock_pool == "HS300":
        stock_list = get_index_stocks("000300.XSHG", feature_date)
    elif stock_pool == "ZZ1000":
        stock_list = get_index_stocks("000852.XSHG", feature_date)
    elif stock_pool == "A":
        stock_list = get_index_stocks("000985.XSHG", feature_date)
    else:
        raise ValueError("unsupported stock_pool: {}".format(stock_pool))

    if len(stock_list) == 0:
        return []

    st_data = get_extras("is_st", stock_list, count=1, end_date=feature_date)
    if st_data is not None and len(st_data) > 0:
        st_row = st_data.iloc[0]
        stock_list = [
            stock for stock in stock_list
            if (stock not in st_row.index) or pd.isnull(st_row[stock]) or (not bool(st_row[stock]))
        ]

    stock_list = filter_paused_stock_by_date(stock_list, feature_date)
    stock_list = delect_stop(stock_list, feature_date, n=MIN_LISTING_DAYS)
    return stock_list


def get_industry_bucket_map_for_data(stock_list, date):
    if len(stock_list) == 0:
        return {}
    try:
        industry_info = get_industry(stock_list, date=date)
    except Exception:
        return {stock: "UNKNOWN" for stock in stock_list}

    out = {}
    for stock in stock_list:
        info = industry_info.get(stock, {})
        bucket = None
        for key in ["sw_l1", "jq_l1", "zjw"]:
            sub = info.get(key, None)
            if isinstance(sub, dict):
                bucket = sub.get("industry_code") or sub.get("industry_name")
                if bucket:
                    break
        out[stock] = bucket if bucket else "UNKNOWN"
    return out


def get_factor_data(stock_list, date):
    if len(stock_list) == 0:
        return pd.DataFrame()

    df_factor = pd.DataFrame(index=stock_list)
    for fac_chunk in chunks(BASE_FACTOR_COLS, 20):
        try:
            factor_data = get_factor_values(
                securities=stock_list,
                factors=fac_chunk,
                count=1,
                end_date=date,
            )
        except Exception:
            factor_data = None

        for fac in fac_chunk:
            try:
                if factor_data is not None and fac in factor_data:
                    df_factor[fac] = factor_data[fac].iloc[0, :]
                else:
                    df_factor[fac] = np.nan
            except Exception:
                df_factor[fac] = np.nan
    return df_factor


def calc_ret(close_mat, days):
    if close_mat is None or close_mat.empty or len(close_mat) <= days:
        return pd.Series(index=close_mat.columns if close_mat is not None else [], dtype=float)
    return close_mat.iloc[-1] / close_mat.iloc[-days - 1] - 1


def calc_up_day_ratio(ret_mat, days):
    if ret_mat is None or ret_mat.empty:
        return pd.Series(dtype=float)
    return (ret_mat.tail(days) > 0).mean()


def calc_new_low_distance(close_mat, days):
    if close_mat is None or close_mat.empty:
        return pd.Series(dtype=float)
    last_close = close_mat.iloc[-1]
    min_close = close_mat.tail(days).min()
    return last_close / min_close - 1


def get_price_path_and_liquidity_data(stock_list, date, lookback=121, chunk_size=160):
    cols = V4_PRICE_PATH_COLS + TRADE_LIQUIDITY_COLS[:9]
    out_all = []
    for stock_chunk in chunks(stock_list, chunk_size):
        out = pd.DataFrame(index=stock_chunk, columns=cols, dtype=float)
        try:
            price_df = get_price(
                stock_chunk,
                end_date=date,
                frequency="daily",
                fields=["close", "high", "low", "volume", "money", "paused"],
                count=lookback,
                skip_paused=False,
                fq="pre",
                panel=False,
                fill_paused=True,
            )
        except Exception:
            price_df = None

        if price_df is None or price_df.empty:
            out_all.append(out)
            continue

        for col in ["close", "high", "low", "volume", "money", "paused"]:
            if col not in price_df.columns:
                price_df[col] = np.nan
        price_df["time"] = pd.to_datetime(price_df["time"]).dt.normalize()
        close_mat = price_df.pivot_table(index="time", columns="code", values="close").sort_index()
        high_mat = price_df.pivot_table(index="time", columns="code", values="high").sort_index()
        low_mat = price_df.pivot_table(index="time", columns="code", values="low").sort_index()
        volume_mat = price_df.pivot_table(index="time", columns="code", values="volume").sort_index()
        money_mat = price_df.pivot_table(index="time", columns="code", values="money").sort_index()
        paused_mat = price_df.pivot_table(index="time", columns="code", values="paused").sort_index()

        ret_mat = close_mat.pct_change()
        last_close = close_mat.iloc[-1]
        ma20 = close_mat.tail(20).mean()
        ma60 = close_mat.tail(60).mean()
        money20 = money_mat.tail(20).mean()
        money60 = money_mat.tail(60).mean()
        volume20 = volume_mat.tail(20).mean()
        volume60 = volume_mat.tail(60).mean()

        out["px_ret_5"] = calc_ret(close_mat, 5)
        out["px_ret_20"] = calc_ret(close_mat, 20)
        out["px_ret_60"] = calc_ret(close_mat, 60)
        out["px_ret_120"] = calc_ret(close_mat, 120)
        out["px_close_to_ma20"] = last_close / ma20 - 1
        out["px_close_to_ma60"] = last_close / ma60 - 1
        out["px_ma20_to_ma60"] = ma20 / ma60 - 1
        out["px_volatility_20"] = ret_mat.tail(20).std()
        out["px_volatility_60"] = ret_mat.tail(60).std()
        out["px_drawdown_20"] = last_close / close_mat.tail(20).max() - 1
        out["px_drawdown_60"] = last_close / close_mat.tail(60).max() - 1
        out["px_drawdown_120"] = last_close / close_mat.tail(120).max() - 1
        out["px_up_day_ratio_20"] = calc_up_day_ratio(ret_mat, 20)
        out["px_new_high_distance_60"] = last_close / close_mat.tail(60).max() - 1
        out["px_new_low_distance_60"] = calc_new_low_distance(close_mat, 60)
        out["px_skew_20"] = ret_mat.tail(20).skew()
        out["px_kurt_20"] = ret_mat.tail(20).kurt()

        out["liq_money_mean_20"] = money20
        out["liq_money_mean_60"] = money60
        out["liq_money_ratio_20_60"] = money20 / money60 - 1
        out["liq_volume_mean_20"] = volume20
        out["liq_volume_ratio_20_60"] = volume20 / volume60 - 1
        out["liq_amplitude_mean_20"] = (high_mat.tail(20) / low_mat.tail(20) - 1).mean()
        out["liq_amplitude_mean_60"] = (high_mat.tail(60) / low_mat.tail(60) - 1).mean()
        out["liq_paused_count_20"] = paused_mat.tail(20).fillna(0).sum()
        out["liq_paused_count_60"] = paused_mat.tail(60).fillna(0).sum()

        out_all.append(out.replace([np.inf, -np.inf], np.nan))
        del price_df, close_mat, high_mat, low_mat, volume_mat, money_mat, paused_mat, ret_mat
        gc.collect()
    return pd.concat(out_all).reindex(index=stock_list)


def get_limit_state_data(stock_list, date, lookback=20, chunk_size=160):
    cols = [
        "liq_low_money_days_20",
        "liq_limit_up_count_20",
        "liq_limit_down_count_20",
        "liq_one_price_limit_count_20",
    ]
    out_all = []
    for stock_chunk in chunks(stock_list, chunk_size):
        out = pd.DataFrame(index=stock_chunk, columns=cols, dtype=float)
        try:
            price_df = get_price(
                stock_chunk,
                end_date=date,
                frequency="daily",
                fields=["close", "high", "low", "money", "paused", "high_limit", "low_limit"],
                count=lookback,
                skip_paused=False,
                fq=None,
                panel=False,
                fill_paused=True,
            )
        except Exception:
            price_df = None

        if price_df is None or price_df.empty:
            out_all.append(out)
            continue

        for col in ["close", "high", "low", "money", "paused", "high_limit", "low_limit"]:
            if col not in price_df.columns:
                price_df[col] = np.nan
        price_df["time"] = pd.to_datetime(price_df["time"]).dt.normalize()
        money_mat = price_df.pivot_table(index="time", columns="code", values="money").sort_index()
        close_mat = price_df.pivot_table(index="time", columns="code", values="close").sort_index()
        high_mat = price_df.pivot_table(index="time", columns="code", values="high").sort_index()
        low_mat = price_df.pivot_table(index="time", columns="code", values="low").sort_index()
        high_limit_mat = price_df.pivot_table(index="time", columns="code", values="high_limit").sort_index()
        low_limit_mat = price_df.pivot_table(index="time", columns="code", values="low_limit").sort_index()

        money_q20 = money_mat.stack().quantile(0.20) if len(money_mat.stack().dropna()) else np.nan
        out["liq_low_money_days_20"] = (money_mat.tail(20) < money_q20).sum() if not pd.isnull(money_q20) else np.nan

        limit_up = close_mat >= (high_limit_mat * 0.999)
        limit_down = close_mat <= (low_limit_mat * 1.001)
        one_price = (high_mat <= low_mat * 1.0001) & (limit_up | limit_down)
        out["liq_limit_up_count_20"] = limit_up.tail(20).sum()
        out["liq_limit_down_count_20"] = limit_down.tail(20).sum()
        out["liq_one_price_limit_count_20"] = one_price.tail(20).sum()

        out_all.append(out.replace([np.inf, -np.inf], np.nan))
        del price_df, money_mat, close_mat, high_mat, low_mat, high_limit_mat, low_limit_mat
        gc.collect()
    return pd.concat(out_all).reindex(index=stock_list)


def get_market_context(date, benchmark=BENCHMARK, lookback=61):
    out = {
        "ctx_market_ret_20": np.nan,
        "ctx_market_ret_60": np.nan,
        "ctx_market_volatility_20": np.nan,
    }
    try:
        bench_df = get_price(
            benchmark,
            end_date=date,
            frequency="daily",
            fields=["close"],
            count=lookback,
            skip_paused=True,
            fq="pre",
        )
    except Exception:
        bench_df = None

    if bench_df is None or bench_df.empty or "close" not in bench_df.columns:
        return out
    close = bench_df["close"].dropna()
    if len(close) > 20:
        out["ctx_market_ret_20"] = close.iloc[-1] / close.iloc[-21] - 1
        out["ctx_market_volatility_20"] = close.pct_change().tail(20).std()
    if len(close) > 60:
        out["ctx_market_ret_60"] = close.iloc[-1] / close.iloc[-61] - 1
    return out


def attach_industry_context(factor_data, market_context):
    out = factor_data.copy()
    for col, value in market_context.items():
        out[col] = value

    for ret_col, ctx_col in [
        ("px_ret_20", "ctx_industry_ret_20"),
        ("px_ret_60", "ctx_industry_ret_60"),
    ]:
        out[ctx_col] = out.groupby("industry_bucket")[ret_col].transform("mean")

    out["ctx_stock_minus_industry_ret_20"] = out["px_ret_20"] - out["ctx_industry_ret_20"]
    out["ctx_stock_minus_industry_ret_60"] = out["px_ret_60"] - out["ctx_industry_ret_60"]
    out["ctx_stock_rank_industry_ret_20"] = out.groupby("industry_bucket")["px_ret_20"].rank(pct=True)
    out["ctx_stock_rank_industry_volatility_20"] = out.groupby("industry_bucket")["px_volatility_20"].rank(pct=True)
    return out


def get_forward_alpha(stock_list, date, next_date, benchmark):
    if len(stock_list) == 0:
        return pd.Series(dtype=float)

    price_df = get_price(
        stock_list,
        start_date=date,
        end_date=next_date,
        frequency="daily",
        fields=["close"],
        skip_paused=True,
        fq="pre",
        panel=False,
    )
    if price_df is None or price_df.empty:
        return pd.Series(dtype=float)

    price_df["time"] = pd.to_datetime(price_df["time"]).dt.normalize()
    close_mat = price_df.pivot_table(index="time", columns="code", values="close").sort_index()
    if len(close_mat) < 2:
        return pd.Series(dtype=float)
    stock_ret = close_mat.iloc[-1] / close_mat.iloc[1] - 1

    bench_df = get_price(
        benchmark,
        start_date=date,
        end_date=next_date,
        frequency="daily",
        fields=["close"],
        skip_paused=True,
        fq="pre",
    )
    if bench_df is None or bench_df.empty or len(bench_df) < 2:
        return pd.Series(dtype=float)

    bench_ret = bench_df["close"].iloc[-1] / bench_df["close"].iloc[1] - 1
    return stock_ret - bench_ret


def add_core_factor_temporal_features(df):
    out = df.copy()
    out = out.sort_values(["rebalance_date", "stock"]).reset_index(drop=True)
    for factor in CORE_TEMPORAL_FACTORS:
        if factor not in out.columns:
            continue
        rank_col = "tmp_{}_rank".format(factor)
        out[rank_col] = out.groupby("rebalance_date")[factor].rank(pct=True)
        g_stock = out.groupby("stock")[rank_col]
        for lag in [1, 3]:
            col = "ts_{}_rank_chg_{}m".format(factor, lag)
            out[col] = out[rank_col] - g_stock.shift(lag)
        mean_col = "ts_{}_rank_mean_3m".format(factor)
        std_col = "ts_{}_rank_std_3m".format(factor)
        z_col = "ts_{}_rank_z_6m".format(factor)
        out[mean_col] = g_stock.transform(lambda s: s.shift(1).rolling(3, min_periods=2).mean())
        out[std_col] = g_stock.transform(lambda s: s.shift(1).rolling(3, min_periods=2).std())
        rolling_mean_6 = g_stock.transform(lambda s: s.shift(1).rolling(6, min_periods=3).mean())
        rolling_std_6 = g_stock.transform(lambda s: s.shift(1).rolling(6, min_periods=3).std())
        out[z_col] = (out[rank_col] - rolling_mean_6) / rolling_std_6
        out = out.drop(columns=[rank_col])
    return out


def build_v46_rebuild_dataset():
    require_joinquant_api()
    date_list = get_period_date("M", V4_DATA_START, V4_DATA_END_FOR_LABEL)
    print("V46 rebuild rebalance dates =", len(date_list), "|", V4_DATA_START, "->", V4_DATA_END_FOR_LABEL)

    all_rows = []
    for i, rebalance_date in enumerate(date_list[:-1]):
        next_date = date_list[i + 1]
        feature_date = get_previous_trade_date(rebalance_date)
        if feature_date is None:
            continue

        stock_list = get_stock(UNIVERSE_NAME, feature_date)
        if len(stock_list) == 0:
            continue

        jq_factor_data = get_factor_data(stock_list, feature_date)
        if jq_factor_data is None or jq_factor_data.empty:
            continue

        industry_map = get_industry_bucket_map_for_data(stock_list, feature_date)
        price_liq_data = get_price_path_and_liquidity_data(stock_list, feature_date)
        limit_data = get_limit_state_data(stock_list, feature_date)
        alpha = get_forward_alpha(stock_list, rebalance_date, next_date, BENCHMARK)
        if alpha.empty:
            continue

        factor_data = jq_factor_data.join(price_liq_data, how="left").join(limit_data, how="left")
        factor_data["stock"] = factor_data.index
        factor_data["industry_bucket"] = factor_data["stock"].map(industry_map).fillna("UNKNOWN")
        factor_data = attach_industry_context(factor_data, get_market_context(feature_date, BENCHMARK))
        factor_data["alpha_1m"] = alpha
        factor_data["rebalance_date"] = rebalance_date
        factor_data["feature_date"] = feature_date
        factor_data["next_date"] = next_date
        factor_data = factor_data.dropna(subset=["alpha_1m"]).copy()
        if len(factor_data) < 30:
            continue

        factor_data["alpha_rank_pct"] = factor_data["alpha_1m"].rank(pct=True, method="first")
        all_rows.append(factor_data.reset_index(drop=True))
        print(
            "  rebuilt {}/{} rebalance={} feature={} rows={}".format(
                i + 1, max(1, len(date_list) - 1), rebalance_date, feature_date, len(factor_data)
            )
        )

        del jq_factor_data, price_liq_data, limit_data, alpha, factor_data
        gc.collect()

    if len(all_rows) == 0:
        raise ValueError("V46 data rebuild produced no rows")
    df = pd.concat(all_rows, ignore_index=True)
    df = add_core_factor_temporal_features(df)
    df.to_csv(V4_DATA_FILE, index=False)
    print("V46 data rebuilt rows =", len(df), "saved ->", V4_DATA_FILE)
    return df





def maybe_rebuild_dataset():
    global RAW_DATA_PATH
    if not REBUILD_DATA:
        print("skip data rebuild; using RAW_DATA_PATH =", RAW_DATA_PATH)
        return None

    if os.path.exists(REBUILD_DATA_OUTPUT_PATH) and not REBUILD_FORCE_OVERWRITE:
        print("rebuilt data already exists; skip rebuild:", REBUILD_DATA_OUTPUT_PATH)
        if USE_REBUILT_DATA_FOR_TRAINING:
            RAW_DATA_PATH = REBUILD_DATA_OUTPUT_PATH
            print("RAW_DATA_PATH switched to existing rebuilt file:", RAW_DATA_PATH)
        return pd.read_csv(REBUILD_DATA_OUTPUT_PATH, nrows=5)

    print("rebuilding CSI800 monthly dataset")
    print("  start =", REBUILD_DATA_START, "end_for_label =", REBUILD_DATA_END_FOR_LABEL)
    rebuilt_df = build_v46_rebuild_dataset()

    for _col in ["rebalance_date", "feature_date", "next_date"]:
        if _col in rebuilt_df.columns:
            rebuilt_df[_col] = pd.to_datetime(rebuilt_df[_col])

    meta_cols = ["stock", "rebalance_date", "feature_date", "next_date", "alpha_1m", "alpha_rank_pct", "industry_bucket"]
    feature_cols_rebuilt = [c for c in rebuilt_df.columns if c not in meta_cols]
    date_min = rebuilt_df["rebalance_date"].min() if len(rebuilt_df) else pd.NaT
    date_max = rebuilt_df["rebalance_date"].max() if len(rebuilt_df) else pd.NaT

    rebuild_manifest = pd.DataFrame([{
        "data_file": REBUILD_DATA_OUTPUT_PATH,
        "rows": int(len(rebuilt_df)),
        "months": int(rebuilt_df["rebalance_date"].nunique()) if "rebalance_date" in rebuilt_df.columns else 0,
        "stock_count": int(rebuilt_df["stock"].nunique()) if "stock" in rebuilt_df.columns else 0,
        "feature_count": int(len(feature_cols_rebuilt)),
        "rebalance_date_min": str(date_min.date()) if not pd.isnull(date_min) else "",
        "rebalance_date_max": str(date_max.date()) if not pd.isnull(date_max) else "",
        "universe": UNIVERSE_NAME,
        "universe_index": UNIVERSE_INDEX,
        "benchmark": BENCHMARK,
        "min_listing_days": MIN_LISTING_DAYS,
        "rebuilt_by": "中证800_V55_anchor_consensus_rerank_monitor实验.ipynb",
    }])
    rebuild_manifest.to_csv(REBUILD_MANIFEST_PATH, index=False)

    print("rebuilt rows =", len(rebuilt_df))
    print("rebalance date =", date_min, "->", date_max)
    print("feature count =", len(feature_cols_rebuilt))
    print("saved data ->", REBUILD_DATA_OUTPUT_PATH)
    print("saved manifest ->", REBUILD_MANIFEST_PATH)
    try:
        display(rebuild_manifest)
    except NameError:
        print(rebuild_manifest)

    if USE_REBUILT_DATA_FOR_TRAINING:
        RAW_DATA_PATH = REBUILD_DATA_OUTPUT_PATH
        print("RAW_DATA_PATH switched to rebuilt file:", RAW_DATA_PATH)

    return rebuilt_df


_rebuilt_preview_df = maybe_rebuild_dataset()
if _rebuilt_preview_df is not None:
    print("rebuild preview shape:", _rebuilt_preview_df.shape)


In [ ]:
# =========================
# Loading, score-panel build, and schema checks
# =========================
def require_columns(df, cols, label):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(label + " missing required columns: " + ",".join(missing))


def normalize_stock_column(df):
    out = df.copy()
    if STOCK_COL not in out.columns:
        if "code" in out.columns:
            out = out.rename(columns={"code": STOCK_COL})
        elif "security" in out.columns:
            out = out.rename(columns={"security": STOCK_COL})
    return out


def normalize_target_columns(df):
    out = df.copy()
    if TARGET_ALPHA_COL not in out.columns and "alpha_vs_csi800" in out.columns:
        out[TARGET_ALPHA_COL] = out["alpha_vs_csi800"]
    if RETURN_COL not in out.columns and "stock_return_1m" in out.columns:
        out[RETURN_COL] = out["stock_return_1m"]
    if RETURN_COL not in out.columns and "raw_return_1m" in out.columns:
        out[RETURN_COL] = out["raw_return_1m"]
    return out


def load_anchor_bundle(path):
    if not os.path.exists(path):
        raise IOError(
            "anchor model pkl not found: " + path + "\n"
            "Upload/copy the anchor pkl to the notebook working directory, or edit ANCHOR_MODEL_PATH."
        )
    with open(path, "rb") as f:
        bundle = pickle.load(f)
    if not isinstance(bundle, dict):
        raise ValueError("anchor pkl should be a dict bundle")
    for key in ["base_model", "base_feature_cols", "base_fill_values"]:
        if key not in bundle:
            raise ValueError("anchor bundle missing key: " + key)
    return bundle


def build_score_panel_from_raw(raw_path, model_path, out_path):
    if not os.path.exists(raw_path):
        raise IOError(
            "score panel not found and raw data not found.\n"
            "score panel: " + out_path + "\n"
            "raw data: " + raw_path + "\n"
            "Put the fixed V46/V410 feature table in RAW_DATA_PATH, or set AUTO_BUILD_SCORE_PANEL=False and provide SCORE_PANEL_PATH."
        )
    raw = pd.read_csv(raw_path)
    raw = normalize_stock_column(raw)
    raw = normalize_target_columns(raw)
    require_columns(raw, [DATE_COL, STOCK_COL, TARGET_ALPHA_COL], "raw data")

    raw[DATE_COL] = pd.to_datetime(raw[DATE_COL])
    raw = raw.replace([np.inf, -np.inf], np.nan)

    bundle = load_anchor_bundle(model_path)
    feature_cols = list(bundle.get("base_feature_cols", []))
    fill_values = dict(bundle.get("base_fill_values", {}))
    missing_features = [c for c in feature_cols if c not in raw.columns]
    if missing_features:
        raise ValueError(
            "raw data missing anchor model features: " + ",".join(missing_features[:30]) +
            (" ..." if len(missing_features) > 30 else "")
        )

    X = raw.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan)
    X = X.fillna(pd.Series(fill_values)).fillna(0.0)
    model = bundle["base_model"]
    best_iter = bundle.get("base_best_iter", None) or bundle.get("model_iter", None) or bundle.get("fixed_iter", None)
    try:
        pred = np.asarray(model.predict(X[feature_cols], num_iteration=best_iter)).reshape(-1)
    except TypeError:
        pred = np.asarray(model.predict(X[feature_cols])).reshape(-1)

    out = raw.copy()
    out[ANCHOR_SCORE_COL] = pred
    out["anchor_model_path"] = model_path
    out["anchor_research_version"] = bundle.get("research_version", "")
    out["anchor_feature_count"] = len(feature_cols)
    out.to_csv(out_path, index=False)
    print("built score panel:", out_path, out.shape)
    print("anchor:", bundle.get("research_version", ""), "features:", len(feature_cols), "iter:", best_iter)
    return out


def load_score_panel(path):
    df = pd.read_csv(path)
    df = normalize_stock_column(df)
    df = normalize_target_columns(df)
    require_columns(df, [DATE_COL, STOCK_COL, ANCHOR_SCORE_COL, TARGET_ALPHA_COL], "score panel")
    df = df.copy()
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])
    df[ANCHOR_SCORE_COL] = pd.to_numeric(df[ANCHOR_SCORE_COL], errors="coerce")
    df[TARGET_ALPHA_COL] = pd.to_numeric(df[TARGET_ALPHA_COL], errors="coerce")
    if RETURN_COL in df.columns:
        df[RETURN_COL] = pd.to_numeric(df[RETURN_COL], errors="coerce")
    if BENCHMARK_COL in df.columns:
        df[BENCHMARK_COL] = pd.to_numeric(df[BENCHMARK_COL], errors="coerce")
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=[DATE_COL, STOCK_COL, ANCHOR_SCORE_COL, TARGET_ALPHA_COL])
    df = df.sort_values([DATE_COL, STOCK_COL]).reset_index(drop=True)
    return df


def load_or_build_score_panel(path):
    if os.path.exists(path):
        print("use existing score panel:", path)
        return load_score_panel(path)
    if not AUTO_BUILD_SCORE_PANEL:
        raise IOError(
            "score panel not found: " + path + "\n"
            "Set AUTO_BUILD_SCORE_PANEL=True, or provide the fixed score panel."
        )
    build_score_panel_from_raw(RAW_DATA_PATH, ANCHOR_MODEL_PATH, path)
    return load_score_panel(path)


def describe_panel(df):
    print("loaded:", df.shape)
    print(df[[DATE_COL]].agg(["min", "max"]))
    print("months:", df[DATE_COL].nunique())
    print("stocks per month:")
    print(df.groupby(DATE_COL)[STOCK_COL].count().describe())
    available_groups = {}
    for group_name, factors in FACTOR_GROUPS.items():
        available_groups[group_name] = [f for f in factors if f in df.columns]
    print("available factor groups:")
    for group_name in sorted(available_groups):
        print("  {}: {}".format(group_name, len(available_groups[group_name])))
    return available_groups

In [ ]:
# =========================
# Scoring helpers
# =========================
def pct_rank_series(s, higher_is_better=True):
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    out = pd.Series(np.nan, index=s.index, dtype=float)
    valid = s.dropna()
    if len(valid) == 0:
        return out
    if len(valid) == 1:
        out.loc[valid.index] = 0.5
        return out
    out.loc[valid.index] = valid.rank(pct=True, method="average", ascending=bool(higher_is_better))
    return out


def add_factor_group_scores(df):
    out = df.copy()
    for group_name in FACTOR_GROUPS:
        out["group_score_" + group_name] = np.nan

    for dt, idx in out.groupby(DATE_COL).groups.items():
        part = out.loc[idx]
        for group_name, factors in FACTOR_GROUPS.items():
            rank_parts = []
            for factor in factors:
                if factor not in part.columns:
                    continue
                higher = FACTOR_DIRECTIONS.get(factor, 1) > 0
                rank_parts.append(pct_rank_series(part[factor], higher))
            if rank_parts:
                group_score = pd.concat(rank_parts, axis=1, sort=False).mean(axis=1)
                out.loc[idx, "group_score_" + group_name] = group_score
    return out


def add_candidate_consensus_scores(df):
    out = df.copy()
    out["is_top{}_candidate".format(TOP_N_CANDIDATES)] = 0
    out["anchor_rank_in_candidate"] = np.nan
    out["vote_count"] = 0
    out["consensus_score"] = np.nan
    out["consensus_defensive_score"] = np.nan
    for group_col in GROUP_SCORE_COLS:
        out[group_col + "_vote"] = 0

    candidate_flag_col = "is_top{}_candidate".format(TOP_N_CANDIDATES)
    risk_cols = [c for c in ["beta", "Variance20", "liquidity", "VOL10", "DAVOL10"] if c in out.columns]

    for dt, idx in out.groupby(DATE_COL).groups.items():
        month = out.loc[idx].copy()
        month = month.dropna(subset=[ANCHOR_SCORE_COL])
        if month.empty:
            continue
        cand = month.sort_values([ANCHOR_SCORE_COL, STOCK_COL], ascending=[False, True]).head(TOP_N_CANDIDATES).copy()
        if cand.empty:
            continue
        cand[candidate_flag_col] = 1
        cand["anchor_rank_in_candidate"] = np.arange(1, len(cand) + 1)
        cand["vote_count"] = 0

        for group_col in GROUP_SCORE_COLS:
            vote_col = group_col + "_vote"
            cand[vote_col] = 0
            if group_col not in cand.columns:
                continue
            valid = cand.dropna(subset=[group_col]).sort_values([group_col, ANCHOR_SCORE_COL, STOCK_COL], ascending=[False, False, True])
            if valid.empty:
                continue
            top_idx = valid.head(min(VOTE_TOP_N, len(valid))).index
            cand.loc[top_idx, vote_col] = 1
            cand["vote_count"] = cand["vote_count"] + cand[vote_col]

        cand["consensus_score"] = cand["vote_count"].astype(float) * 100.0 - cand["anchor_rank_in_candidate"].astype(float)

        if risk_cols:
            risk_parts = []
            for c in risk_cols:
                # Lower risk/liquidity-crowding ranks are better here.
                risk_parts.append(pct_rank_series(cand[c], higher_is_better=False))
            risk_quality = pd.concat(risk_parts, axis=1, sort=False).mean(axis=1)
            cand["consensus_defensive_score"] = cand["consensus_score"] + 10.0 * risk_quality.fillna(0.0)
        else:
            cand["consensus_defensive_score"] = cand["consensus_score"]

        write_cols = [candidate_flag_col, "anchor_rank_in_candidate", "vote_count", "consensus_score", "consensus_defensive_score"]
        write_cols.extend([c + "_vote" for c in GROUP_SCORE_COLS])
        out.loc[cand.index, write_cols] = cand[write_cols]

    return out

In [ ]:
# =========================
# Portfolio evaluation
# =========================
def compound_return(s):
    s = pd.Series(s).dropna().astype(float)
    if len(s) == 0:
        return np.nan
    return float((1.0 + s).prod() - 1.0)


def max_drawdown_from_returns(s):
    s = pd.Series(s).dropna().astype(float)
    if len(s) == 0:
        return np.nan
    curve = (1.0 + s).cumprod()
    peak = curve.cummax()
    dd = curve / peak - 1.0
    return float(dd.min())


def select_month_portfolio(month, score_col, n, candidate_only):
    if candidate_only:
        candidate_flag_col = "is_top{}_candidate".format(TOP_N_CANDIDATES)
        month = month[month[candidate_flag_col] == 1].copy()
    else:
        month = month.copy()
    month = month.dropna(subset=[score_col])
    if month.empty:
        return month
    sort_cols = [score_col]
    ascending = [False]
    if score_col != ANCHOR_SCORE_COL and ANCHOR_SCORE_COL in month.columns:
        sort_cols.append(ANCHOR_SCORE_COL)
        ascending.append(False)
    sort_cols.append(STOCK_COL)
    ascending.append(True)
    return month.sort_values(sort_cols, ascending=ascending).head(n).copy()


def evaluate_one_profile(df, strategy_name, score_col, n, candidate_only):
    rows = []
    use_return_col = RETURN_COL if RETURN_COL in df.columns else TARGET_ALPHA_COL
    use_benchmark = BENCHMARK_COL if BENCHMARK_COL in df.columns else None
    for dt, month in df.groupby(DATE_COL):
        selected = select_month_portfolio(month, score_col, n, candidate_only)
        if selected.empty:
            continue
        ret = selected[use_return_col].mean()
        alpha = selected[TARGET_ALPHA_COL].mean()
        benchmark = month[use_benchmark].dropna().iloc[0] if use_benchmark is not None and month[use_benchmark].notnull().any() else np.nan
        excess = ret - benchmark if use_benchmark is not None and pd.notnull(benchmark) else alpha
        row = {
            "rebalance_date": dt,
            "strategy_name": strategy_name,
            "score_col": score_col,
            "target_count": int(len(selected)),
            "return_1m": float(ret),
            "alpha_1m": float(alpha),
            "benchmark_1m": float(benchmark) if pd.notnull(benchmark) else np.nan,
            "excess_1m": float(excess),
            "targets": ",".join(selected[STOCK_COL].astype(str).tolist()),
            "mean_anchor_rank_in_candidate": selected["anchor_rank_in_candidate"].mean() if "anchor_rank_in_candidate" in selected.columns else np.nan,
            "mean_vote_count": selected["vote_count"].mean() if "vote_count" in selected.columns else np.nan,
        }
        if BOARD_COL in selected.columns:
            row["boards"] = ",".join(selected[BOARD_COL].astype(str).tolist())
        if INDUSTRY_COL in selected.columns:
            row["industries"] = ",".join(selected[INDUSTRY_COL].astype(str).tolist())
        rows.append(row)
    if rows:
        return pd.DataFrame(rows).sort_values(["strategy_name", "rebalance_date"]).reset_index(drop=True)
    return pd.DataFrame()


def build_monthly_results(df):
    parts = []
    for n in PORTFOLIO_SIZES:
        parts.append(evaluate_one_profile(df, "anchor_top{}".format(n), ANCHOR_SCORE_COL, n, False))
        parts.append(evaluate_one_profile(df, "consensus_top{}".format(n), "consensus_score", n, True))
        parts.append(evaluate_one_profile(df, "defensive_consensus_top{}".format(n), "consensus_defensive_score", n, True))
    parts = [p for p in parts if p is not None and not p.empty]
    if not parts:
        return pd.DataFrame()
    monthly = pd.concat(parts, ignore_index=True, sort=False)
    monthly = monthly.sort_values(["strategy_name", "rebalance_date"]).reset_index(drop=True)
    monthly["cum_return"] = monthly.groupby("strategy_name")["return_1m"].transform(lambda s: (1.0 + s).cumprod() - 1.0)
    monthly["cum_excess"] = monthly.groupby("strategy_name")["excess_1m"].transform(lambda s: (1.0 + s).cumprod() - 1.0)
    return monthly


def summarize_monthly(monthly):
    rows = []
    if monthly.empty:
        return pd.DataFrame()
    for name, gdf in monthly.groupby("strategy_name"):
        excess = gdf["excess_1m"].astype(float)
        returns = gdf["return_1m"].astype(float)
        sorted_excess = excess.sort_values(ascending=False).reset_index(drop=True)
        rows.append({
            "strategy_name": name,
            "months": int(len(gdf)),
            "cum_return": compound_return(returns),
            "cum_excess": compound_return(excess),
            "mean_monthly_return": float(returns.mean()),
            "mean_monthly_excess": float(excess.mean()),
            "win_rate_excess": float((excess > 0).mean()),
            "max_drawdown_return": max_drawdown_from_returns(returns),
            "max_drawdown_excess": max_drawdown_from_returns(excess),
            "drop_top1_excess": compound_return(sorted_excess.iloc[1:]) if len(sorted_excess) > 1 else np.nan,
            "drop_top3_excess": compound_return(sorted_excess.iloc[3:]) if len(sorted_excess) > 3 else np.nan,
            "avg_target_count": float(gdf["target_count"].mean()),
            "avg_vote_count": float(gdf["mean_vote_count"].mean()) if "mean_vote_count" in gdf.columns else np.nan,
        })
    return pd.DataFrame(rows).sort_values("cum_excess", ascending=False).reset_index(drop=True)

In [ ]:
# =========================
# Diagnostics
# =========================
def calc_spearman(x, y):
    tmp = pd.DataFrame({"x": x, "y": y}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(tmp) < 5:
        return np.nan
    return float(tmp["x"].rank().corr(tmp["y"].rank()))


def build_health_monitor(df):
    rows = []
    for dt, month in df.groupby(DATE_COL):
        ic = calc_spearman(month[ANCHOR_SCORE_COL], month[TARGET_ALPHA_COL])
        top10 = month.sort_values([ANCHOR_SCORE_COL, STOCK_COL], ascending=[False, True]).head(10)
        top30 = month.sort_values([ANCHOR_SCORE_COL, STOCK_COL], ascending=[False, True]).head(TOP_N_CANDIDATES)
        realized_top10 = month.sort_values([TARGET_ALPHA_COL, STOCK_COL], ascending=[False, True]).head(10)
        realized_top20 = month.sort_values([TARGET_ALPHA_COL, STOCK_COL], ascending=[False, True]).head(20)
        top30_names = set(top30[STOCK_COL].astype(str).tolist())
        top10_names = set(top10[STOCK_COL].astype(str).tolist())
        realized10_names = set(realized_top10[STOCK_COL].astype(str).tolist())
        realized20_names = set(realized_top20[STOCK_COL].astype(str).tolist())
        rows.append({
            "rebalance_date": dt,
            "rank_ic": ic,
            "anchor_top10_alpha": float(top10[TARGET_ALPHA_COL].mean()) if len(top10) else np.nan,
            "anchor_top30_alpha": float(top30[TARGET_ALPHA_COL].mean()) if len(top30) else np.nan,
            "top30_recall_realized_top10": float(len(top30_names.intersection(realized10_names))) / 10.0 if len(realized10_names) else np.nan,
            "top30_recall_realized_top20": float(len(top30_names.intersection(realized20_names))) / 20.0 if len(realized20_names) else np.nan,
            "top10_hit_realized_top10": float(len(top10_names.intersection(realized10_names))) / 10.0 if len(realized10_names) else np.nan,
        })
    health = pd.DataFrame(rows).sort_values("rebalance_date").reset_index(drop=True)
    if health.empty:
        return health
    health["rank_ic_roll6"] = health["rank_ic"].rolling(RECENT_WINDOW_MONTHS, min_periods=3).mean()
    health["top30_recall_top10_roll6"] = health["top30_recall_realized_top10"].rolling(RECENT_WINDOW_MONTHS, min_periods=3).mean()
    health["anchor_top10_alpha_roll6"] = health["anchor_top10_alpha"].rolling(RECENT_WINDOW_MONTHS, min_periods=3).mean()
    health["health_flag"] = "normal"
    weak = (health["rank_ic_roll6"] < 0.0) | (health["top30_recall_top10_roll6"] < 0.20) | (health["anchor_top10_alpha_roll6"] < 0.0)
    health.loc[weak.fillna(False), "health_flag"] = "weak_signal"
    return health


def build_topk_recall(df):
    rows = []
    for dt, month in df.groupby(DATE_COL):
        for candidate_k in [10, 20, 30, 50]:
            pred = month.sort_values([ANCHOR_SCORE_COL, STOCK_COL], ascending=[False, True]).head(candidate_k)
            pred_names = set(pred[STOCK_COL].astype(str).tolist())
            for realized_k in [5, 10, 20]:
                realized = month.sort_values([TARGET_ALPHA_COL, STOCK_COL], ascending=[False, True]).head(realized_k)
                real_names = set(realized[STOCK_COL].astype(str).tolist())
                rows.append({
                    "rebalance_date": dt,
                    "candidate_k": candidate_k,
                    "realized_k": realized_k,
                    "overlap_count": len(pred_names.intersection(real_names)),
                    "recall": float(len(pred_names.intersection(real_names))) / float(realized_k) if realized_k > 0 else np.nan,
                })
    return pd.DataFrame(rows)


def build_vote_attribution(df):
    candidate_flag_col = "is_top{}_candidate".format(TOP_N_CANDIDATES)
    cand = df[df[candidate_flag_col] == 1].copy()
    if cand.empty:
        return pd.DataFrame()
    rows = []
    for vote_count, gdf in cand.groupby("vote_count"):
        rows.append({
            "vote_count": int(vote_count),
            "rows": int(len(gdf)),
            "months": int(gdf[DATE_COL].nunique()),
            "mean_alpha_1m": float(gdf[TARGET_ALPHA_COL].mean()),
            "median_alpha_1m": float(gdf[TARGET_ALPHA_COL].median()),
            "mean_anchor_rank_in_candidate": float(gdf["anchor_rank_in_candidate"].mean()),
        })
    out = pd.DataFrame(rows).sort_values("vote_count", ascending=False).reset_index(drop=True)
    return out


def build_latest_targets(df):
    if df.empty:
        return pd.DataFrame()
    latest = df[DATE_COL].max()
    month = df[df[DATE_COL] == latest].copy()
    rows = []
    for n in PORTFOLIO_SIZES:
        specs = [
            ("anchor_top{}".format(n), ANCHOR_SCORE_COL, False),
            ("consensus_top{}".format(n), "consensus_score", True),
            ("defensive_consensus_top{}".format(n), "consensus_defensive_score", True),
        ]
        for name, score_col, candidate_only in specs:
            selected = select_month_portfolio(month, score_col, n, candidate_only)
            rows.append({
                "rebalance_date": latest,
                "strategy_name": name,
                "score_col": score_col,
                "target_count": int(len(selected)),
                "mean_vote_count": float(selected["vote_count"].mean()) if "vote_count" in selected.columns and len(selected) else np.nan,
                "targets": ",".join(selected[STOCK_COL].astype(str).tolist()) if len(selected) else "",
            })
    return pd.DataFrame(rows)

In [ ]:
# =========================
# Run experiment
# =========================
score_df = load_or_build_score_panel(SCORE_PANEL_PATH)
available_groups = describe_panel(score_df)

scored_df = add_factor_group_scores(score_df)
scored_df = add_candidate_consensus_scores(scored_df)

monthly_df = build_monthly_results(scored_df)
summary_df = summarize_monthly(monthly_df)
health_df = build_health_monitor(scored_df)
topk_recall_df = build_topk_recall(scored_df)
vote_attribution_df = build_vote_attribution(scored_df)
latest_targets_df = build_latest_targets(scored_df)

scored_df.to_csv(os.path.join(OUT_DIR, "v55_scored_panel.csv"), index=False)
monthly_df.to_csv(os.path.join(OUT_DIR, "v55_monthly.csv"), index=False)
summary_df.to_csv(os.path.join(OUT_DIR, "v55_summary.csv"), index=False)
health_df.to_csv(os.path.join(OUT_DIR, "v55_health_monitor.csv"), index=False)
topk_recall_df.to_csv(os.path.join(OUT_DIR, "v55_topk_recall.csv"), index=False)
vote_attribution_df.to_csv(os.path.join(OUT_DIR, "v55_vote_attribution.csv"), index=False)
latest_targets_df.to_csv(os.path.join(OUT_DIR, "v55_latest_targets.csv"), index=False)

print("summary:")
print(summary_df)
print("\nrecent health tail:")
print(health_df.tail(12))
print("\nvote attribution:")
print(vote_attribution_df)
print("\nlatest targets:")
print(latest_targets_df)

## 结论填写区

运行后只按下面三类判断，不要只看累计收益：

1. `consensus_top10/top20` 是否同时不弱于 `anchor_top10/top20`。
2. `top30_recall_realized_top10` 是否足够高。如果 anchor top30 本身 recall 不够，rerank 很难救；如果 recall 很高但 top10 命中差，rerank 才有空间。
3. `vote_attribution` 是否单调：高 vote_count 的真实 `alpha_1m` 应该明显更高，否则 vote 只是噪声。

保留条件：共识排序至少在 top10/top20 两个 profile 的月度路径、回撤、drop-top-month stress 上不劣于 anchor，并且 vote attribution 有经济解释。

放弃条件：只赢一个 profile、只赢少数月份、或高 vote_count 没有更高 alpha。